# Prepare Eastern Victoria gravimeter data

The Geosoft XYZ data format file is commonly used in geophysics to transmit geophysical survey data in a text-readable form.

This tutorial demonstrates the preparation of XYZ airborne gravity data for QC.

We need all the data in HDF5 geoWhizz format because all the QC functions expect that format. (More on the geoWhizz format elsewhere in the `AirGravQC` documentation.)

___

First, import the required modules, and set the path to the geowhizz files.

In [ ]:
from pathlib import Path

import AirGravQC as qc

This is all very much step by step to illustrate the process, and you can certainly compress some of these steps in your own work.

In [ ]:
# Our survey data consists of 4 XYZ files, all in the same directory.
data_root = r'./EastVicData/'

# The name of the input file for the survey data.
EastVicXYZ_file = Path(data_root + r'EastVic.xyz')

# For the EastVic survey, we have test line data in a separate XYZ file ...
EastVicTestXYZ_file = Path(data_root + r'EastVicTest.xyz')

# ... and another XYZ file with the repeat line data.
EastVicRepXYZ_file = Path(data_root + r'EastVicRepeats.xyz')

# Finally, we also have the planned drape file, or plan file.
# ... sometimes, but not this time, the plan data is in a different directory.
plan_root = data_root
EastVicXYZ_plan = Path(plan_root + r'EastVicPlan.xyz')

___

**The Survey Plan Data**

It is best to convert the plan data to geowhizz format first since some information in the plan data can be used to better prepare the survey data.

In [ ]:
# Convert the plan data to XYZ format
EastVicHDF_plan = qc.xyzToHDF(Path(EastVicXYZ_plan), projectName='EastVic')

In [ ]:
# Add in some meta-data (used in plot titles)
block_name = 'Survey Plan'
qc.updateProject(EastVicHDF_plan, acquirer='Sander Geophysics', blockID=block_name)

In [ ]:
# Set meta-data in the CoordFrame group. The `x`, `y`, and `alt` are the names of
# the default x, y, z channels used in many QC functions.
qc.updateCoordFrame(EastVicHDF_plan,
                    x='W84_UTM_55S_X_m', 
                    y='W84_UTM_55S_Y_m', 
                    alt='Drape_m', 
                    geoDatum='WGS84', 
                    projection='UTM', 
                    utmz='55')

In [ ]:
# Set the attributes for each channel. The `units` are used in plot labels and in some QC analysis
# so should always be set if known. The `descriptions` are rarely used, so definitely optional.
qc.updateChannelAttributes(EastVicHDF_plan, 'W84_UTM_55S_X_m', units='m')
qc.updateChannelAttributes(EastVicHDF_plan, 'W84_UTM_55S_Y_m', units='m')
qc.updateChannelAttributes(EastVicHDF_plan, 'Drape_m', units='m', description='Height relative to WGS84')

In [ ]:
# Summary report of the newly created geowhizz plan data file.
qc.reportWhizz(EastVicHDF_plan)

___

**The Measured Survey Data**

In [ ]:
EastVicHDF_file = qc.xyzToHDF(Path(EastVicXYZ_file), projectName='EastVic')

In [ ]:
# The block name appears on some plots, so it is useful to set it so that it indicates for which data the plots are shown.
block_name = 'EastVic Field Data'
qc.updateProject(EastVicHDF_file, acquirer='Sander Geophysics', blockID=block_name)

In [ ]:
# In measured data, we can also name the default time channel.
qc.updateCoordFrame(EastVicHDF_file, 
                    lat='LAT', 
                    lon='LONG', 
                    x='MGA_X', 
                    y='MGA_Y', 
                    time='FTIME', 
                    alt='MGA_Z',
                    geoDatum='GDA2020', 
                    htDatum='GRS80', 
                    projection='MGA', 
                    utmz='55')

In [ ]:
# The `line_type` tells the code how to extract the planned line number, the segment number, and the repeat number
# from each reported survey line number. This is necessary for comparing positioning of measured data aginst planned
# positions. Flight numbers and dates are recorded as line meta-data in the geowhizz file.

# More details on Line Types can be found elsewhere in the documentation.

qc.updateLineAttributes(EastVicHDF_file, planfiles=EastVicHDF_plan, line_type='SGL_GA', flight_chan='FLIGHT', date_chan="DOY")

In [ ]:
qc.updateChannelAttributes(EastVicHDF_file, 'FTIME', units='s')
qc.updateChannelAttributes(EastVicHDF_file, 'MGA_X', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'MGA_Y', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'MGA_Z', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'MSL_Z', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'LAT', units='degree')
qc.updateChannelAttributes(EastVicHDF_file, 'LONG', units='degree')
qc.updateChannelAttributes(EastVicHDF_file, 'DEM', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'RALT', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'LALT', units='m')
qc.updateChannelAttributes(EastVicHDF_file, 'FX', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FY', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FZ', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'V_EAST', units='m/s')
qc.updateChannelAttributes(EastVicHDF_file, 'V_NORTH', units='m/s')
qc.updateChannelAttributes(EastVicHDF_file, 'EOTCOR', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'LATCOR', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'STATCOR', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'ATMCOR', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FACOR_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FACOR_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'TACOR', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FA56s_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FA56s_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FA100s_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'FA100s_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'B56s_267_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'B56s_267_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'B100s_267_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicHDF_file, 'B100s_267_GEOID', units='mGal')

In [ ]:
qc.reportWhizz(EastVicHDF_file)

In [ ]:
# The report functions allow you to look in more detail at specific elements within the data.
qc.reportWhizz(EastVicHDF_file, line='2341.500', channel='ATMCOR')

In [ ]:
# Setting the `detailed` flag to True provides additional information.
qc.reportFlights(EastVicHDF_file, detailed=True)

In [ ]:
qc.reportSampling(EastVicHDF_file)

___

**The Measured Test-Line Data**

The test line data have all the same properties as the survey data, so we repeat all the same steps, but for different input and output files.
And we change the block name.

<div class="alert alert-block alert-warning">
WARNING - 
The supplier used underscores in the channel names for the survey data (coordinates and Bouguer gravity), but dashes for the repeat and test line data (e.g. `MGA_X` vs `MGA-X`). So we must get that right for `updateCoordFrame` and `updateChannelAttributes`.
<div>

In [ ]:
EastVicTestHDF_file = qc.xyzToHDF(EastVicTestXYZ_file, projectName='EastVic')

In [ ]:
block_name = 'EastVic Test Line Data'
qc.updateProject(EastVicTestHDF_file, acquirer='Sander Geophysics', blockID=block_name)
qc.updateCoordFrame(EastVicTestHDF_file, 
                    lat='LAT', 
                    lon='LONG', 
                    x='MGA-X', 
                    y='MGA-Y', 
                    time='FTIME', 
                    alt='MGA-Z',
                    geoDatum='GDA2020', 
                    htDatum='GRS80', 
                    projection='MGA', 
                    utmz='55')

In [ ]:
qc.updateLineAttributes(EastVicTestHDF_file, planfiles=EastVicHDF_plan, line_type='SGL_GA', flight_chan='FLIGHT', date_chan="DOY")

In [ ]:
qc.updateChannelAttributes(EastVicTestHDF_file, 'FTIME', units='s')
qc.updateChannelAttributes(EastVicTestHDF_file, 'MGA-X', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'MGA-Y', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'MGA-Z', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'MSL-Z', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'LAT', units='degree')
qc.updateChannelAttributes(EastVicTestHDF_file, 'LONG', units='degree')
qc.updateChannelAttributes(EastVicTestHDF_file, 'DEM', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'RALT', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'LALT', units='m')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FX', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FY', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FZ', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'V_EAST', units='m/s')
qc.updateChannelAttributes(EastVicTestHDF_file, 'V_NORTH', units='m/s')
qc.updateChannelAttributes(EastVicTestHDF_file, 'EOTCOR', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'LATCOR', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'STATCOR', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'ATMCOR', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FACOR_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FACOR_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'TACOR', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FA56s_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FA56s_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FA100s_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'FA100s_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'B56s-267_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'B56s-267_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'B100s-267_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicTestHDF_file, 'B100s-267_GEOID', units='mGal')

In [ ]:
qc.reportWhizz(EastVicTestHDF_file)

In [ ]:
qc.reportFlights(EastVicTestHDF_file, detailed=False)

In [ ]:
qc.reportSampling(EastVicTestHDF_file)

___

**The Measured Repeat-Line Data**

In [ ]:
EastVicRepHDF_file = qc.xyzToHDF(EastVicRepXYZ_file, projectName='EastVic')

In [ ]:
block_name = 'EastVic Repeat Line Data'
qc.updateProject(EastVicRepHDF_file, acquirer='Sander Geophysics', blockID=block_name)
qc.updateCoordFrame(EastVicRepHDF_file, lat='LAT', lon='LONG', x='MGA-X', y='MGA-Y', time='FTIME', alt='MGA-Z')
qc.updateCoordFrame(EastVicRepHDF_file, geoDatum='GDA2020', htDatum='GRS80', projection='MGA', utmz='55')

In [ ]:
qc.updateLineAttributes(EastVicRepHDF_file, planfiles=EastVicHDF_plan, line_type='SGL_GA', flight_chan='FLIGHT', date_chan="DOY")

In [ ]:
qc.updateChannelAttributes(EastVicRepHDF_file, 'FTIME', units='s')
qc.updateChannelAttributes(EastVicRepHDF_file, 'MGA-X', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'MGA-Y', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'MGA-Z', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'MSL-Z', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'LAT', units='degree')
qc.updateChannelAttributes(EastVicRepHDF_file, 'LONG', units='degree')
qc.updateChannelAttributes(EastVicRepHDF_file, 'DEM', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'RALT', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'LALT', units='m')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FX', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FY', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FZ', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'V_EAST', units='m/s')
qc.updateChannelAttributes(EastVicRepHDF_file, 'V_NORTH', units='m/s')
qc.updateChannelAttributes(EastVicRepHDF_file, 'EOTCOR', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'LATCOR', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'STATCOR', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'ATMCOR', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FACOR_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FACOR_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'TACOR', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FA56s_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FA56s_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FA100s_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'FA100s_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'B56s-267_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'B56s-267_GEOID', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'B100s-267_GRS80', units='mGal')
qc.updateChannelAttributes(EastVicRepHDF_file, 'B100s-267_GEOID', units='mGal')

In [ ]:
qc.reportWhizz(EastVicRepHDF_file)

In [ ]:
qc.reportFlights(EastVicRepHDF_file)

In [ ]:
qc.reportSampling(EastVicRepHDF_file)

___

**Make a survey flight-line map**

In [ ]:
qc.linesMap([EastVicHDF_file], whizzPlanFile=EastVicHDF_plan)

The test and repeat line numbers do not appear in the plan file, and so they are not shown in the line mape above. But if we do not provide a plan file, then `linesMap` will plot them.

In [ ]:
qc.linesMap([EastVicRepHDF_file, EastVicTestHDF_file])